# Run: play a game on the real robot (or in Gazebo)

**Milestone:** "Complete a full game with no human intervention" (README, proposed: 10 games).
Wires the real vision source and arm driver into `GameManager`, logs to `logging.dir`.

**TODO checklist**
- [ ] Library notebooks implemented (`game_loop.ipynb` and everything it runs).
- [ ] `RealVision`: adapter around gambit_vision (camera -> board assembly -> BoardObservation).
- [ ] `RealArm`: adapter around gambit_arm pick/place (or gambit_sim teleport in Gazebo).
- [ ] Choose the robot policy: `EnginePolicy` (strength-limited) or the teacher.
- [ ] Record per-move success / retries for the README measurement logs.

In [ ]:
%run ./game_loop.ipynb

In [ ]:
from pathlib import Path

REPO = Path("../..").resolve()
LOG_DIR = REPO / GAME_CFG["logging"]["dir"]
HUMAN_COLOR = chess.WHITE if GAME_CFG["game"]["human_color"] == "white" else chess.BLACK
USE_TEACHER = False          # True: gambit_teach TeacherPolicy (needs a player model)

## Adapters (TODO)

In [ ]:
class RealVision:
    """VisionSource on the real camera."""

    def read(self) -> BoardObservation:
        """One camera reading as a BoardObservation."""
        # Design: call gambit_vision's board reader (capture frame, rectify with the ArUco
        # homography, classify 64 squares) and fill placement, per-square confidence, timestamp,
        # hand_present.
        # TODO: implement
        raise NotImplementedError


class RealArm:
    """ArmInterface on the SO-ARM101 (gambit_arm) or Gazebo (gambit_sim teleport).
    TODO: forward each primitive to the arm package and return its success flag."""

    def pick(self, square): raise NotImplementedError
    def place(self, square): raise NotImplementedError
    def move_piece(self, from_sq, to_sq): raise NotImplementedError
    def capture_to_graveyard(self, square): raise NotImplementedError
    def place_from_graveyard(self, piece, square): raise NotImplementedError
    def park(self): raise NotImplementedError

## Play

In [ ]:
# Step 1: build a strength-limited player engine (make_engine, ENGINE_CFG default_elo) and a
#         full-strength analyst engine for logging evals.
# Step 2: choose the policy: EnginePolicy(engine), or if USE_TEACHER
#         `%run ../../gambit_teach/notebooks/move_selection.ipynb` and build a TeacherPolicy with a
#         loaded PlayerModel.
# Step 3: GameLogger(LOG_DIR) + HesitationTracker; GameManager(RealVision(), RealArm(), policy,
#         HUMAN_COLOR, listeners=[tracker, make_log_listener(logger, analyst)]).
# Step 4: gm.run() inside try/finally: always finish the log (result or "*") and close both engines.
# Step 5: print the result and the PGN path; copy per-move success / retries into the README logs.